# Gemma 4 12B LoRA — 原初种族新闻评论（Kaggle 免费 T4）

Before you run anything:
1. Upload `lora_dataset.zip` as a **private** Kaggle Dataset. Go to Datasets → New Dataset; any name works.
2. Notebook settings (right-hand panel):
   - **Accelerator: GPU T4 x2.**
   - **Internet: On.** This needs a phone-verified account.
   - **Add Input:** the dataset from step 1.
3. Click **Save Version → Save & Run All (Commit)**. The run continues in the
   background for up to 12 hours, so you can close the browser.

Training takes about 4–6 h for 2 epochs on 1043 examples. Evaluating 20
examples takes about 1 h more.

The adapter is saved to `/kaggle/working/out/adapter`. Download it from the
version's Output tab.

If the run is cut off, add that version's Output as an input and set
`RESUME = True` below. Training then continues from the last checkpoint,
saved every 50 steps.

In [ ]:
RESUME = False      # True: continue from a previous version's checkpoints
EPOCHS = 2
EVAL_N = 20         # eval examples to generate after training (~3 min each on a T4)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
%%capture
!pip install -q --upgrade unsloth unsloth_zoo

In [ ]:
# Only the training scripts; the repo's news.db and pages are not needed.
!rm -rf /tmp/primalrace && git clone -q --depth 1 --filter=blob:none --no-checkout https://github.com/superpower55/primalrace /tmp/primalrace
%cd /tmp/primalrace
!git sparse-checkout set --no-cone '/news/lora/' '/news/prompts/' '/news/export_lora_dataset.py' '/news/db.py' && git checkout -q
!ls news news/lora

In [ ]:
import glob, os, shutil
os.makedirs('news/dataset/lora', exist_ok=True)
for name in ('train.jsonl', 'eval.jsonl', 'stats.json'):
    hits = glob.glob(f'/kaggle/input/**/{name}', recursive=True)
    assert hits, f'{name} not found under /kaggle/input — did you add the dataset as input?'
    shutil.copy(hits[0], f'news/dataset/lora/{name}')
    print(name, '<-', hits[0])
# Resuming: bring the previous version's checkpoints back
OUT = '/kaggle/working/out'
if RESUME:
    for ck in glob.glob('/kaggle/input/**/out/checkpoint-*', recursive=True):
        dst = os.path.join(OUT, os.path.basename(ck))
        if not os.path.exists(dst):
            shutil.copytree(ck, dst)
    print('checkpoints:', sorted(os.listdir(OUT)) if os.path.isdir(OUT) else 'none')

In [ ]:
# Token counts and the answer mask (no GPU needed)
!python3 news/lora/train_lora.py --check-only --max-seq 7168

In [ ]:
resume = '--resume' if RESUME else ''
!python3 news/lora/train_lora.py --out {OUT} --epochs {EPOCHS} --max-seq 7168 --save-steps 50 {resume}

In [ ]:
# Free the training memory before generating
import gc, torch; gc.collect(); torch.cuda.empty_cache()
!cd news/lora && python3 generate_local.py --adapter {OUT}/adapter -n {EVAL_N} --max-seq 7168 \
    --out /kaggle/working/results/lora.replies.jsonl

In [ ]:
!python3 news/lora/eval_lora.py --replies /kaggle/working/results/lora.replies.jsonl --tag lora -n {EVAL_N}
!cp -r news/dataset/lora/results/* /kaggle/working/results/ 2>/dev/null; ls -la /kaggle/working/results

In [ ]:
# Read a few replies
import json
for line in list(open('/kaggle/working/results/lora.replies.jsonl'))[:3]:
    r = json.loads(line)
    print(f"#{r['id']}  {r['seconds']}s\n{r['raw'][:1500]}\n" + '-' * 80)

Next steps:
- Download `out/adapter` and `results/` from the Output tab.
- Converting to GGUF for Ollama needs about 30 GB RAM and 30 GB disk. Kaggle
  won't fit that, so do it elsewhere; see `news/lora/README.md`.